## 0 · Setup prostředí

Workbench běží na RHOAI _Standard Data Science Notebook_ image. Pandas,
numpy, scikit-learn, matplotlib, boto3 a od RHOAI 3.2 i **MLflow klient**
(Red Hat build, viz [KCS 7136121](https://access.redhat.com/articles/7136121))
jsou už předinstalované. Žádný `pip install` se v notebooku nedělá.

Platforma navíc workbench podu sama nastavuje `MLFLOW_TRACKING_URI`,
`MLFLOW_TRACKING_AUTH=kubernetes-namespaced` a namountuje SA token —
tracking server je tím pádem k mání bez jediného řádku konfigurace.

In [ ]:
# Ověř, že workbench je naladěn proti platform-managed MLflow.
# RHOAI 3.4 přidá tyhle env vars automaticky, když je `mlflowoperator`
# zapnutý v DataScienceCluster (kontrola: `oc get dsc -A`).
import os, mlflow
print(f'mlflow client      : {mlflow.__version__}')
print(f'MLFLOW_TRACKING_URI: {os.environ.get("MLFLOW_TRACKING_URI", "(unset)")}')
print(f'MLFLOW_TRACKING_AUTH: {os.environ.get("MLFLOW_TRACKING_AUTH", "(unset)")}')

# 01 · Průzkum faktur — pohled controllera

**Scénář ze slide 20:** _self-service pro analytiky_ · _rychlejší prototypování_

Tento notebook simuluje situaci, kdy controller dostane úkol "podívej se,
jestli v posledních fakturách nejsou nějaké podezřelé položky".
Bez čekání na BI tým si otevře workbench na Red Hat AI, načte data z MinIO
a během minut má první obrázek, kde hledat.

Žádný kód do produkce — jen rychlý průzkum, který by jinak zabral týden.

## 1.1 Připojení k datům

Workbench používá _Data Connection_ — RHOAI nám pomocí proměnných prostředí
předá přístup k MinIO bucketu, kde leží exportované faktury z Pohody.

In [ ]:
import os, sys
sys.path.insert(0, '../src')  # umožní import sdílených helperů

from invoice_features import load_invoices

# When running on RHOAI workbench, Data Connection sets these env vars.
# Locally we fall back to the bundled CSV so the notebook is portable.
if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import get_s3_client, get_bucket
    s3 = get_s3_client()
    df = load_invoices('invoices.csv', s3_client=s3, bucket=get_bucket())
    print(f'Načteno z S3 bucketu {get_bucket()}')
else:
    df = load_invoices('../data/invoices.csv')
    print('Načteno z lokálního CSV (mimo cluster)')

print(f'Řádků: {len(df):,}')
df.head()

## 1.2 Co máme v datech

Sloupce `is_anomaly` a `anomaly_type` jsou v reálu samozřejmě **neznámé** —
tady je máme jen proto, abychom si na konci ověřili, že model najde to,
co skutečně vadí. V produkci je tam neuvidí ani analytik, ani model.

In [ ]:
df.dtypes

In [ ]:
# Quick overview — kolik faktur za kategorii, kolik celkem proteklo
summary = (df
    .groupby('category')
    .agg(faktur=('invoice_id', 'count'),
         celkem_czk=('amount_czk', 'sum'),
         prum_czk=('amount_czk', 'mean'))
    .sort_values('celkem_czk', ascending=False))
summary

## 1.3 Distribuce částek

Klasická rychlá kontrola: log-stupnice, ať uvidíme i extrémní hodnoty.
Distribuce je **vícemodálová** — to není anomálie, ale projev toho, že
každá nákladová kategorie má svůj typický rozsah (kancl ≠ marketing ≠
subdodávka). Z grafu by se daly chytit jen výrazné outliery vlevo a
vpravo; většina podezřelých faktur (kulaté částky, víkendové doklady,
chybějící PO) ale leží **uvnitř módů** a v 1D histogramu je neuvidíme.

Proto v notebooku 02 trénujeme model nad celým feature space —
kombinace signálů, ne jediná osa.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(df['amount_czk']), bins=60, color='#cc0000', alpha=0.7)
ax.set_xlabel('log10(amount CZK)')
ax.set_ylabel('počet faktur')
ax.set_title('Distribuce výše faktur (log10)')
plt.tight_layout(); plt.show()

## 1.4 První náhled na podezřelé řádky

Než pustíme model, podívejme se očima controllera na nejjednodušší red flags:

- faktura **bez PO** s vysokou částkou,
- **kulatá** částka (50 000, 100 000, …),
- doklad zaúčtovaný o **víkendu** nebo svátku.

Tyhle pravidla bychom napsali i v Excelu. Pointa demo je, že model najde
i ty kombinace, na které žádné jednoduché pravidlo nedosáhne.

In [ ]:
import pandas as pd

df['issued_dow'] = pd.to_datetime(df['issued_on']).dt.day_name()

rule_hits = df[
    ((df['po_number'].fillna('') == '') & (df['amount_czk'] > 100_000))
    | (df['amount_czk'] % 10_000 == 0)
    | (df['issued_dow'].isin(['Saturday', 'Sunday']))
]
print(f'Pravidlové red-flag řádky: {len(rule_hits)} z {len(df)} ({len(rule_hits)/len(df):.1%})')
rule_hits.head(10)

## 1.5 Co tím controller získal

Za pět buněk a několik minut má:

- napojení na živá data (žádný ticket na IT),
- první rozdělení podle dodavatele a kategorie,
- shortlist na základě pravidel.

V dalším notebooku si analytik natrénuje model, který tyhle signály
**kombinuje** a najde i to, co jednoduchá pravidla minou.